This part of the pipeline constructs the pangenomes of the genome sets using MMseqs2 via the interface of PPanGGolin.

### Checking dependencies

In [ ]:
conda activate ppanggolin
ppanggolin --version
conda deactivate

### Paths and parameters

#### Pipeline input folders

In [ ]:
indices_folder='02-GTDB/subgroups'

#### Pipeline output folders

In [ ]:
task_root="03-pangenomes"
mkdir -p $task_root

#### Tool pointers and parameters

In [ ]:
no_cores=22
identity=0.4
coverage=0.4

### Construct the pangenomes

In [ ]:
indices=$(find $indices_folder -type f | grep -E '\.list$' | cut -d '.' -f 1 | xargs -I % basename %)

In [ ]:
echo $indices

In [ ]:
function construct_pangenome() {
    mkdir -p $4
    conda activate ppanggolin
    ppanggolin annotate -f -d --anno $1 -o $4 -c $no_cores
    ppanggolin cluster -f -d -p $4/pangenome.h5 --identity $2 --coverage $3 -c $no_cores
    ppanggolin write_pangenome -f -d -p $4/pangenome.h5 -o $4 --csv --Rtab
    ppanggolin fasta -f -d -p $4/pangenome.h5 -o $4 --prot_families all
    conda deactivate
}

In [ ]:
for group in $indices
do
echo $group
construct_pangenome $indices_folder/$group.list $identity $coverage $task_root/$group
done